# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the [FAIR<sup>2</sup>](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset and its metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available `RecordSet` entries, their fields, and unique `@id` values for consistent reference.

In [ ]:
# List available record sets by @id and name
record_sets = metadata.record_sets

print(f"This dataset contains {len(record_sets)} record sets.\n")
for rset in record_sets:
    print(f"Record Set Name: {rset.name}\n  @id: {rset.id}\n  Description: {rset.description if hasattr(rset, 'description') else ''}")
    if hasattr(rset, 'fields'):
        print("  Fields:")
        for fld in rset.fields:
            print(f"    Field: {fld.name}, @id: {fld.id}, dataType: {fld.data_type if hasattr(fld, 'data_type') else ''}")
    print("")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Reference record sets and fields by their `@id`s.

In [ ]:
# Gather all record set @ids
record_set_ids = [rset.id for rset in metadata.record_sets]

print("Record set IDs:", record_set_ids)

dataframes = {}
for rset_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rset_id))
        if records:
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
        dataframes[rset_id] = df
        print(f"Loaded {len(df)} records from Record Set '{rset_id}' with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load RecordSet {rset_id}: {e}")

# Pick the main tabular record set (often called 'Proteins' or similar in mass spec datasets)
# We'll select the one with 'protein' or 'summary' in the @id as an example.
main_rset_id = None
for rid in record_set_ids:
    if ('protein' in rid.lower()) or ('summary' in rid.lower()):
        main_rset_id = rid
        break
if not main_rset_id:
    # default to first
    main_rset_id = record_set_ids[0]

print(f"\nChosen main Record Set: {main_rset_id}\nAvailable columns: {dataframes[main_rset_id].columns.tolist()}")
dataframes[main_rset_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records by a numeric field, normalizing, and grouping. Use field `@id`s for reference.


In [ ]:
# Display all available fields for EDA
df = dataframes[main_rset_id]
print(f"Columns in main Record Set ({main_rset_id}): \n{df.columns.tolist()}")

# Try to auto-select a numeric field by checking dtypes or suggest some common ones
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if not numeric_cols:
    # heuristic: look for columns containing 'count', 'abundance', 'mw', 'coverage', 'score', 'intensity', or 'peptide'
    for col in df.columns:
        if any(keyword in col.lower() for keyword in ['count', 'abundance', 'mw', 'coverage', 'score', 'intensity', 'peptide', 'number', 'value']):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                continue
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Selected numeric field for EDA: {numeric_field}")
else:
    print("No numeric field found for EDA.")
    numeric_field = None

# Filtering, normalizing, grouping
if numeric_field is not None:
    threshold = df[numeric_field].mean()  # Example threshold: above-average values
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered rows with {numeric_field} > {threshold}")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
    )
    print(f"\nNormalized {numeric_field} values:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical column (find string columns with few unique values)
    cat_cols = [c for c in df.select_dtypes(include=['object', 'category']).columns if df[c].nunique() < 15 and c != numeric_field]
    if cat_cols:
        group_field = cat_cols[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}', showing mean {numeric_field}:")
        print(grouped_df.head())
    else:
        group_field = None
else:
    group_field = None

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR<sup>2</sup> dataset on human mass spectrometry extracellular vesicles. We identified available record sets and fields by their `@id`, loaded the main record set, and demonstrated common preprocessing and analysis tasks, such as filtering by abundance or counts, normalizing data, and group-wise aggregation.

The `mlcroissant` library enables a robust, schema-driven workflow for inspecting and preparing FAIR-compliant datasets. For further study, consider exploring relationships between other fields, integrating with proteomics tools, or exporting curated records for downstream statistical analysis.
